# My Pipeline for part 2

## Using argpase

In [ ]:
import argparse

import argparse

def main():
    parser = argparse.ArgumentParser(description="Simulate NGS reads from a reference genome.")
    parser.add_argument("fasta_file", help="Input FASTA file name (e.g., EcoliK12-MG1655.fasta)")
    parser.add_argument("--outdir", default="data/processed", help="Output directory for FASTQ file")
    parser.add_argument("--snps", type=int, default=300, help="Number of SNPs to introduce")
    parser.add_argument("--indels", type=int, default=10, help="Number of INDELs to introduce")
    parser.add_argument("--max_indel_len", type=int, default=10, help="Maximum length of an INDEL")
    parser.add_argument("--read_len", type=int, default=100, help="Average fragment/read length")
    # Add other parameters (e.g., read depth, quality score)

    args = parser.parse_args()

    # The rest of your pipeline logic would use args.fasta_file, args.snps, etc.
    # ... your existing code goes here, using 'args' variables ...

if __name__ == "__main__":
    main()

    



# Put this into run_pipeline.py

In [ ]:
# run_pipeline.py

import os
import argparse
# Import functions
from functions.parse_fasta import parse_fasta
from functions.make_mutations import make_mutations
from functions.get_depth import get_depth
from functions.create_fastq import create_fastq

def run_pipeline(args):
    """Executes the pipeline."""

    # 1. Parse FASTA Input
    print(f"Parsing FASTA: {args.fasta_file}...")
    file_path = os.path.join("data", "raw", args.fasta_file)
    sequence_to_mutate_dict = parse_fasta(file_path)

    # Use the first sequence in the dictionary for mutation (as per your original script)
    fasta_id, sequence_to_mutate = list(sequence_to_mutate_dict.items())[args.which_fasta]
    database_tag = fasta_id.strip().split(" ", maxsplit=1)
    print(f"Target Sequence: {database_tag[0]}")

    # 2. Make Mutations
    print(f"Introducing {args.snps} SNPs and {args.indels} INDELs...")
    mutated_sequence = make_mutations(
        sequence_to_mutate, args.snps, args.indels, args.max_indel_len
    )

    # 3. Get Fragmented Reads (Simulate Sequencing Depth)
    print(f"Generating fragmented reads with avg length {args.read_len}...")
    # NOTE: Your get_depth takes a hardcoded '30' for 'read_coverage', which should also be an argument
    read_coverage = 30 # For now, keep it hardcoded or add to argparse
    mutated_fragments = get_depth(mutated_sequence, read_coverage, args.read_len)

    # 4. Create FASTQ Output
    fastq_filename = database_tag[0] + ".fastq"
    os.makedirs(args.outdir, exist_ok=True) # Ensure output directory exists
    file_path_and_name = os.path.join(args.outdir, fastq_filename)
    
    print(f"Writing FASTQ file to: {file_path_and_name}")
    # Assume perfect reads for simplicity
    quality_score_ascii = "~" 
    
    create_fastq(
        file_path_and_name, database_tag, args.which_fasta, mutated_fragments, quality_score_ascii
    )
    print("Pipeline complete!")


def main():
    parser = argparse.ArgumentParser(description="Simulate NGS reads from a reference genome.")
    parser.add_argument("fasta_file", help="Input FASTA file name (e.g., EcoliK12-MG1655.fasta)")
    parser.add_argument("--outdir", default="data/processed", help="Output directory for FASTQ file. Default: data/processed")
    parser.add_argument("--snps", type=int, default=300, help="Number of SNPs to introduce. Default: 300")
    parser.add_argument("--indels", type=int, default=10, help="Number of INDELs to introduce. Default: 10")
    parser.add_argument("--max_indel_len", type=int, default=10, help="Maximum length of an INDEL. Default: 10")
    parser.add_argument("--read_len", type=int, default=100, help="Average fragment/read length. Default: 100")
    parser.add_argument("--which_fasta", type=int, default=0, help="Index of the sequence in the FASTA file to use. Default: 0 (first entry)")
    
    args = parser.parse_args()
    run_pipeline(args)

if __name__ == "__main__":
    main()

# Pipeline without argparser

In [ ]:
sequence_filename = "EcoliK12-MG1655.fasta"

#Parse fasta
from functions.parse_fasta import parse_fasta
import os
file_path = os.path.join("data", "raw", sequence_filename)

sequence_to_mutate_dict = parse_fasta(file_path)
print(sequence_to_mutate_dict)

sequence_details = {}
fasta_id = []
sequence_to_mutate = ""
description_then_sequence = []

for fasta_id, sequence in sequence_to_mutate_dict.items():
    fasta_id = fasta_id.replace(". ", ".") #formatting fasta_id
    sequence_to_mutate = sequence
    fasta_details = fasta_id.strip().split(" ", maxsplit=1)
    # fasta_details structure: seq_id, genus species, strain, sub-strain, description
description_then_sequence.append(fasta_details[1])
description_then_sequence.append(sequence_to_mutate)
sequence_details[fasta_details[0]] = description_then_sequence
# key is NCBI tag
# value is description, then sequence

which_fasta = 0 # Modifiable: which fasta file to run? index = 0 (1st)

database_tag = fasta_details

print(database_tag[which_fasta])
print(sequence[:30])

# Make mutations
no_snp = 300
no_indel = 10
max_indel_length = 10

from functions.make_mutations import make_mutations
mutated_sequence = make_mutations(sequence_to_mutate, no_snp, no_indel, max_indel_length)
# parameters: sequence, no_snp, no_indel, max_indel_length

# Get depth
mutated_fragments = []
avg_fragment_length = 100
from functions.get_depth import get_depth
mutated_fragments = get_depth(mutated_sequence, 30, avg_fragment_length)

# Transfer fragmented reads into fastq file perfect phred score ~
quality_score_ascii = "~" #Assume perfect reads

fastq_location = "data/processed/"
fastq_file_extension = ".fastq"
file_path_and_name = fastq_location + database_tag[which_fasta] + fastq_file_extension

from functions.create_fastq import create_fastq
create_fastq(file_path_and_name, database_tag, which_fasta, mutated_fragments, quality_score_ascii)
# parameters: file path, tag of start sequence, which fasta, mutated reads, quality score ascii